# OpenDQI Python quickstart

Three patterns in 3 minutes — same as `examples/python/{01,02,03}_*.py`, but executable cell-by-cell.

**Install:** `pip install opendqi` (v0.12.1+).

**This notebook expects** the OpenDQI repo cloned locally so the synthetic fixtures under `examples/` are reachable. For a stand-alone tutorial that downloads its own fixtures, see `docs/python.md`.

In [1]:
import json, os, shutil, subprocess
from pathlib import Path

import opendqi
print('opendqi version:', opendqi.__version__)

# Repo root (notebook is at examples/python/quickstart.ipynb)
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'Cargo.toml').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
print('repo root:', REPO_ROOT)

opendqi version: 0.12.0
repo root: /Users/paul/Desktop/opendqi


## Pattern 1 — Scan a normalized Parquet

The simplest entry point: hand `scan_parquet` a path to a canonical EMIR Parquet (produced by `opendqi emir normalize`). Get back a `summary` dict and an `issues` pyarrow.Table.

Here we generate the Parquet on the fly via the CLI binary.

In [2]:
csv = REPO_ROOT / 'examples' / 'emir' / 'sample.csv'
mapping = REPO_ROOT / 'examples' / 'emir' / 'sample_mapping.yml'
parquet = REPO_ROOT / 'target' / 'examples' / 'sample.parquet'
parquet.parent.mkdir(parents=True, exist_ok=True)

if not parquet.exists():
    bin_path = shutil.which('opendqi') or str(REPO_ROOT / 'target' / 'debug' / 'opendqi')
    subprocess.run(
        [bin_path, 'emir', 'normalize', str(csv),
         '--mapping', str(mapping), '--out', str(parquet)],
        check=True, env={**os.environ, 'RUST_LOG': 'warn'},
    )

result = opendqi.emir.scan_parquet(str(parquet))
print(json.dumps(result.summary, indent=2, default=str))

{
  "regime": "emir",
  "files_processed": 1,
  "records_processed": 8,
  "issues_total": 97,
  "issues_by_severity": {
    "warning": 58,
    "high": 37,
    "critical": 2
  },
  "issues_by_dimension": {
    "completeness": 50,
    "uniqueness": 2,
    "timeliness": 8,
    "validity": 25,
    "accuracy": 10,
    "consistency": 2
  },
  "quality_score": 25.75,
  "started_at": "2026-05-20T17:26:42.191022+00:00",
  "finished_at": "2026-05-20T17:26:42.193781+00:00"
}


## Pattern 2 — XML → `scan_table` (no Parquet roundtrip)

When you already have ISO 20022 XML in memory, `parse_xml` produces the canonical Arrow Table directly; `scan_table` then runs the same check suite as `scan_parquet`.

In [3]:
xml = REPO_ROOT / 'examples' / 'quickstart-emir' / 'auth030-tar.xml'

table = opendqi.emir.parse_xml(str(xml))
print(f'parsed {table.num_rows} record(s), {len(table.column_names)} columns')

mapping = {n: n for n in table.column_names}   # identity
result = opendqi.emir.scan_table(table, mapping)
print(json.dumps(result.summary, indent=2, default=str))

parsed 20 record(s), 55 columns
{
  "regime": "emir",
  "files_processed": 1,
  "records_processed": 20,
  "issues_total": 197,
  "issues_by_severity": {
    "warning": 86,
    "high": 106,
    "critical": 5
  },
  "issues_by_dimension": {
    "completeness": 180,
    "uniqueness": 5,
    "consistency": 12
  },
  "quality_score": 27.849998474121094,
  "started_at": "2026-05-20T17:26:42.369539+00:00",
  "finished_at": "2026-05-20T17:26:42.369699+00:00"
}


## Pattern 3 — Custom column mapping (the warehouse path)

When your Arrow table comes from a custom source, the column names won't match the canonical EMIR field names. The `mapping` dict reroutes each canonical field to your actual column name.

Below: a small Arrow groupby on `result.issues` showing the top 5 check IDs — pure Arrow, no pandas needed.

In [4]:
rename = {
    'uti':                 'TradeUTI',
    'valuation_timestamp': 'MtmTs',
    'maturity_date':       'ContractEnd',
}
user_table = table.rename_columns([rename.get(n, n) for n in table.column_names])

mapping = {
    **{name: name for name in user_table.column_names if name not in rename.values()},
    'uti':                 'TradeUTI',
    'valuation_timestamp': 'MtmTs',
    'maturity_date':       'ContractEnd',
}
result = opendqi.emir.scan_table(user_table, mapping)
print(f'records={result.summary["records_processed"]} '
      f'issues={result.summary["issues_total"]} '
      f'score={result.summary["quality_score"]:.2f}')

from collections import Counter
top_checks = Counter(result.issues.column('check_id').to_pylist()).most_common(5)
print('\nTop 5 check IDs:')
for check_id, n in top_checks:
    print(f'  {n:>3}x {check_id}')

records=20 issues=197 score=27.85

Top 5 check IDs:
   20x EMIR.COMP.ASSET_CLASS_MISSING
   20x EMIR.COMP.CLEARING_STATUS_MISSING
   20x EMIR.COMP.COUNTERPARTY_1_MISSING
   20x EMIR.COMP.COUNTERPARTY_2_MISSING
   20x EMIR.COMP.INTRAGROUP_INDICATOR_MISSING


## Where to go next

- [`docs/python.md`](../../docs/python.md) — full quickstart with DuckDB / Polars / pandas / Spark integration patterns
- [`docs/python-roadmap.md`](../../docs/python-roadmap.md) — architecture spec + v0.13+ roadmap
- [`README.md`](../../README.md) — project overview, CLI + UI surface, 216 checks coverage

**Status: preview (v0.12.x).** The v1.0 Arrow contract for `result.issues` (11 cols) is **stable**; the API surface may grow additively in v0.13 (`scan_directory`, `tr_audit`, Spark UDF namespace).